### Set up

In [99]:
#import matplotlib
import matplotlib.pyplot as plt
#import pathlib
import mne
#import mne_bids
import numpy as np
import pandas as pd
import os
from pyprep.find_noisy_channels import NoisyChannels
#from functions import mark_bad_channels
#import autoreject

In [ ]:
print(mne.__version__)
print(pyprep.__version__)

In [4]:
%matplotlib qt
#%matplotlib inline

In [5]:
bdf_file_path = '/home/p2894/mne_eeg_workshop/ds006777/sub-1501/eeg/sub-1501_task-AVSRT_run-01_eeg.bdf'

In [100]:
subject_id = 'sub-1501'
base_dir = '/home/p2894/mne_eeg_workshop/ds006777/derivatives' # 請替換為您的主要專案路徑
subject_folder = os.path.join(base_dir, subject_id)

# 3. 自動建立資料夾（如果已經存在則忽略，不會報錯）
os.makedirs(subject_folder, exist_ok=True)

In [102]:
log_file_path = os.path.join(subject_folder, 'sub-1501_log.txt')
mne.set_log_file(log_file_path, overwrite=True)

In [18]:
def mark_bad_channels(raw):
    raw = raw.copy()
    raw.filter(l_freq=1.0, h_freq=45.0)
    nd = NoisyChannels(raw)
    nd.find_all_bads(ransac=True, channel_wise=True) # Call all the functions to detect bad channels.
    bads = nd.get_bads() # Get the names of all channels currently flagged as bad. Returns bads
    return bads # list or dict of bad channels

### Import bdf

In [20]:
raw = mne.io.read_raw_bdf(bdf_file_path, preload=True)
raw
# https://mne.tools/stable/generated/mne.io.read_raw_bdf.html
# Notes about importing: https://mne.tools/stable/auto_tutorials/io/20_reading_eeg_data.html
    # It is advisable to choose a reference after importing BioSemi data to avoid losing signal information.

<RawBDF | sub-1501_task-AVSRT_run-01_eeg.bdf, 73 x 449536 (878.0 s), ~250.4 MiB, data loaded>

### Events

In [21]:
events = mne.find_events(raw, shortest_event=1)
# Find events from raw file.
# https://mne.tools/stable/generated/mne.find_events.html

In [22]:
events_id = {
    "AV": 3,
    "A": 4,
    "V": 5,
    "Response": 1
}

In [49]:
# rename events to plot
from mne.viz import plot_events
mne.viz.plot_events(events, sfreq=raw.info['sfreq']);

### Montage

In [ ]:
#mne.channels.get_builtin_montages()
# Get a list of all standard montages shipping with MNE-Python. The names of the montages can be passed to make_standard_montage().
# https://mne.tools/stable/generated/mne.channels.get_builtin_montages.html

In [23]:
montage = mne.channels.make_standard_montage('biosemi64')
# Read a generic (built-in) standard montage that ships with MNE-Python.
# https://mne.tools/stable/generated/mne.channels.make_standard_montage.html

In [ ]:
montage.plot()  # 2D
fig = montage.plot(kind="3d", show=False)  # 3D
fig = fig.gca().view_init(azim=70, elev=15)  # set view angle for tutorial
# https://mne.tools/stable/auto_tutorials/intro/40_sensor_locations.html#Working%20with%20built-in%20montages

In [24]:
raw = raw.set_montage(montage, on_missing='ignore')
# Set channel positions and digitization points.
# https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.set_montage

In [25]:
#raw = raw.set_channel_types()
# Specify the sensor types of channels.
# https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.set_channel_types

raw.set_channel_types({"EXG1":"emg"})
raw.set_channel_types({"EXG2":"emg"})
raw.set_channel_types({"EXG3":"emg"})
raw.set_channel_types({"EXG4":"emg"})
raw.set_channel_types({"EXG5":"emg"})
raw.set_channel_types({"EXG6":"emg"})
raw.set_channel_types({"EXG7":"emg"})
raw.set_channel_types({"EXG8":"emg"})

<RawBDF | sub-1501_task-AVSRT_run-01_eeg.bdf, 73 x 449536 (878.0 s), ~250.4 MiB, data loaded>

### Bad channel detection

In [26]:
bads = mark_bad_channels(raw)
raw.info['bads'] = bads
raw.info['bads']

['PO8', 'O1', 'P4', 'O2', 'P9', 'Pz', 'POz', 'T7', 'Oz']

In [27]:
if len(raw.info['bads']) > len(raw.get_channel_types(picks=['eeg']))*0.15:
    print('The data should be discarded.')

### Re-reference

In [28]:
raw_ref = raw.set_eeg_reference(ref_channels='average')
# Bad EEG channels are automatically excluded if they are properly set in info['bads'].

### Create two independent copies of `raw_ref`

In [29]:
# Create two independent copies of raw_ref to prevent simultaneous object mutation
raw_for_ica = raw_ref.copy()
raw_for_analysis = raw_ref.copy()

### ICA

In [30]:
# Preparation for ICA
raw_for_ica = raw_for_ica.copy().filter(l_freq=1.0, h_freq=40.0)

In [31]:
n_components = None
random_state = 42
method = 'fastica'
fit_params = None
max_iter = 1000

ica = mne.preprocessing.ICA(n_components=n_components, method=method, max_iter=max_iter, fit_params=fit_params, random_state=random_state)
# https://mne.tools/stable/generated/mne.preprocessing.ICA.html
# https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html

In [32]:
picks_ica = mne.pick_types(raw_for_ica.info, eeg=True, eog=False, exclude="bads") # array of int: indices of good channels
ica.fit(raw_for_ica, picks=picks_ica)

Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=1000
Fit,88 iterations on raw data (449536 samples)
ICA components,54
Available PCA components,55
Channel types,eeg
ICA components marked for exclusion,—


In [ ]:
eog_evoked = mne.preprocessing.create_eog_epochs(raw_for_ica.copy(), ch_name=['Fp1', 'Fp2']).average()
# https://mne.tools/stable/generated/mne.preprocessing.create_eog_epochs.html
eog_evoked.apply_baseline(baseline=(None, -0.2))
eog_evoked.plot_joint();

In [33]:
ica.exclude = []
eog_indices, eog_scores = ica.find_bads_eog(raw_for_ica.copy(), ch_name=['Fp1', 'Fp2'])
ica.exclude = eog_indices

In [34]:
ica.exclude

[np.int64(0), np.int64(2)]

In [35]:
ica.apply(raw_for_analysis)

<RawBDF | sub-1501_task-AVSRT_run-01_eeg.bdf, 73 x 449536 (878.0 s), ~250.4 MiB, data loaded>

### Filter

In [36]:
raw_for_analysis = raw_for_analysis.copy().filter(l_freq=0.05, h_freq=40)
# https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.filter

### Rename events with bad responses

In [ ]:
# 沒反應
# < 100 ms
# > 1000 ms
# 反應兩次以上

In [ ]:
sfreq = raw.info['sfreq']

events_news = events.copy()
events_news = events_news[np.where((events_news[:,2]==3) | (events_news[:,2]==4) | (events_news[:,2]==5) | (events_news[:,2]==1))]

# 準備一個空的列表，用來收集好 trial 的資料
valid_rt_data = []

# 建立一個字典，方便把代碼轉換成好懂的字串
stim_dict = {3: 'AV', 4: 'A', 5: 'V'}

for i in range(len(events_news)):
    current_event = events_news[i, 2]
    
    # 步驟 1：只針對刺激事件 (3, 4, 5) 進行檢查
    if current_event in [3, 4, 5]:
        
        response_count = 0  # 用來計算按鍵次數
        rt = 0.0            # 用來記錄反應時間
        
        # 步驟 2：開啟「往後找」的迴圈，尋找直到下一個刺激出現前的所有反應
        # i + 1 確保從下一個事件開始找，不會越界
        for j in range(i + 1, len(events_news)):
            next_event = events_news[j, 2]
            
            # 如果撞到下一個刺激 (3, 4, 5)，代表這個 trial 結束了，停止尋找
            if next_event in [3, 4, 5]:
                break
                
            # 如果找到按鍵反應 (1)
            elif next_event == 1:
                # 如果是「第一次」按鍵，記錄下反應時間 (後面的時間 - 前面的時間)
                if response_count == 0:
                    rt = (events_news[j, 0] - events_news[i, 0]) / sfreq
                
                # 按鍵次數 +1
                response_count += 1
                
        # 步驟 3：結算這個 trial 是否犯規
        is_invalid = False
        
        if response_count == 0:
            is_invalid = True  # 犯規：無反應
        elif response_count > 1:
            is_invalid = True  # 犯規：按了兩次以上
        else:
            # 只有剛好按 1 次的情況，才需要檢查反應時間
            if rt < 0.1 or rt > 1.0:
                is_invalid = True  # 犯規：太快或太慢
                
        # 步驟 4：如果犯規，把標記乘上 10 (3->30, 4->40, 5->50)
        if is_invalid:
            events_news[i, 2] = current_event * 10
        else:
            # 好 trial：把資料打包成字典，存進我們的列表中
            valid_rt_data.append({
                'Event_Index': i,                   # 記錄它在 events_news 中的位置 (方便除錯)
                'Stimulus_Type': stim_dict[current_event], # 轉換成 'AV', 'A', 'V'
                'Reaction_Time': rt                 # 記錄算出來的反應時間 (秒)
            })

In [69]:
events_epochs = events_news[np.where((events_news[:,2]==3) | (events_news[:,2]==4) | (events_news[:,2]==5) | (events_news[:,2]==30) | (events_news[:,2]==40) | (events_news[:,2]==50))]

In [78]:
events_news[:10]

array([[1359,    0,   40],
       [1954,    0,    1],
       [2929,    0,    4],
       [3093,    0,    1],
       [3790,    0,    5],
       [4150,    0,    1],
       [4933,    0,    4],
       [5178,    0,    1],
       [6076,    0,    5],
       [6329,    0,    1]])

In [82]:
events_epochs

array([[  1359,      0,     40],
       [  2929,      0,      4],
       [  3790,      0,      5],
       ...,
       [445982,      0,      5],
       [446698,      0,      3],
       [448302,      0,      3]], shape=(400, 3))

### Epoching & baseline correction & PTP rejection

In [83]:
event_dict = {
    "AV": 3,
    "A": 4,
    "V": 5
}

In [84]:
epochs = mne.Epochs(raw_for_analysis, events=events_epochs, event_id=event_dict, tmin=-0.5, tmax=0.8, baseline=(-0.2, 0), reject=dict(eeg=200e-6), detrend=0, preload=False)
# Epochs extracted from a Raw instance.
# https://mne.tools/stable/generated/mne.Epochs.html

In [90]:
epochs.drop_log

(('IGNORED',),
 (),
 (),
 ('Fp1', 'Fpz', 'Fp2', 'AF8', 'AF4'),
 ('Fpz',),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('AF7',
  'F5',
  'CP1',
  'P7',
  'F2',
  'F8',
  'FT8',
  'FC6',
  'FC4',
  'FC2',
  'FCz',
  'C6',
  'P2',
  'P10'),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('PO4',),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 ('IGNORED',),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 (),
 ('IGNORED',),
 ('IGNORED',),
 (),

### Behavioral analysis

In [80]:
# 迴圈結束後，把收集到的好 trial 列表轉換成 Pandas DataFrame
df_valid_rts = pd.DataFrame(valid_rt_data)

# 顯示前幾筆資料看看成果
print(df_valid_rts.head())

   Event_Index Stimulus_Type  Reaction_Time
0            2             A       0.320312
1            4             V       0.703125
2            6             A       0.478516
3            8             V       0.494141
4           10             A       0.474609


In [81]:
df_valid_rts.groupby('Stimulus_Type')['Reaction_Time'].mean()

Stimulus_Type
A     0.360979
AV    0.294568
V     0.374225
Name: Reaction_Time, dtype: float64

In [85]:
print(len(np.where((events_news[:,2]==3))[0])) # AV good
print(len(np.where((events_news[:,2]==30))[0])) # AV bad
print(len(np.where((events_news[:,2]==4))[0])) # A good
print(len(np.where((events_news[:,2]==40))[0])) # A bad
print(len(np.where((events_news[:,2]==5))[0])) # V good
print(len(np.where((events_news[:,2]==50))[0])) # V bad

127
4
123
4
131
11


In [91]:
# 1. 計算各條件的 Good 與 Bad 數量
# AV (視聽雙通道)
av_good = len(np.where(events_news[:, 2] == 3)[0])
av_bad  = len(np.where(events_news[:, 2] == 30)[0])
av_total = av_good + av_bad
# 避免除以零的保護機制：如果總數大於 0 就計算，否則給予空值 (NaN)
av_hit_rate = (av_good / av_total) if av_total > 0 else np.nan

# A (聽覺單通道)
a_good = len(np.where(events_news[:, 2] == 4)[0])
a_bad  = len(np.where(events_news[:, 2] == 40)[0])
a_total = a_good + a_bad
a_hit_rate = (a_good / a_total) if a_total > 0 else np.nan

# V (視覺單通道)
v_good = len(np.where(events_news[:, 2] == 5)[0])
v_bad  = len(np.where(events_news[:, 2] == 50)[0])
v_total = v_good + v_bad
v_hit_rate = (v_good / v_total) if v_total > 0 else np.nan

In [97]:
# 先建立基礎表格
data = {
    'Condition': ['AV', 'A', 'V'],
    'Good_Trials': [av_good, a_good, v_good],
    'Bad_Trials': [av_bad, a_bad, v_bad],
    'Total_Trials': [av_total, a_total, v_total],
    'Hit_Rate': [av_hit_rate, a_hit_rate, v_hit_rate]
}
df_summary = pd.DataFrame(data)

# 轉換成百分比字串（方便閱讀）
df_summary['Hit_Rate_Percent'] = (df_summary['Hit_Rate'] * 100).round(2).astype(str) + '%'


# ==========================================
# 2. 新增步驟：把 Mean RT 打包進來！
# ==========================================
# 先計算出各組的平均反應時間 (這會得到一個以 'AV', 'A', 'V' 為索引的 Series)
mean_rts = df_valid_rts.groupby('Stimulus_Type')['Reaction_Time'].mean()

# 利用 map() 功能，根據 'Condition' 欄位名稱 ('AV', 'A', 'V') 對應填入平均值
# 小數點後四捨五入到第 3 位 (毫秒等級)
df_summary['Mean_RT_Secs'] = df_summary['Condition'].map(mean_rts) #.round(3)

In [98]:
df_summary

,Condition,Good_Trials,Bad_Trials,Total_Trials,Hit_Rate,Hit_Rate_Percent,Mean_RT_Secs
0,AV,127,4,131,0.969466,96.95%,0.294568
1,A,123,4,127,0.968504,96.85%,0.360979
2,V,131,11,142,0.922535,92.25%,0.374225


### Bad channel interpolation (to be considered)

In [ ]:
#epochs_interpolate = epochs.interpolate_bads()
# https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.interpolate_bads
# https://mne.tools/stable/auto_examples/preprocessing/interpolate_bad_channels.html

### Save files to derivatives

In [101]:
beh_file_path = os.path.join(subject_folder, 'sub-1501_beh.csv')
df_summary.to_csv(beh_file_path, index=False)

eeg_file_path = os.path.join(subject_folder, 'sub-1501_eeg-epo.fif')
epochs.save(eeg_file_path, overwrite = True)

[PosixPath('/home/p2894/mne_eeg_workshop/ds006777/derivatives/sub-1501/sub-1501_eeg-epo.fif')]